# Experiment 16 — Label-noise sensitivity (MIMIC)

The MIMIC CheXpert NLP labels are imperfect. This experiment builds a
high-confidence MIMIC subset by intersecting CheXpert positives with a
radiologist-report regex per disease, then re-fits the linear probe on the
intersection. Preservation of the between-model ranking on the
high-confidence subset indicates that the reported dissociation is not a
label-noise artifact.


In [ ]:
# === Papermill parameters (override via `papermill -p NAME VALUE`) ===
DATASET = "nih"            # one of: nih, mimic, emory
MODELS = "raddino,dinov2,dinov3,biomedclip,medsiglip"
SEED = 42
OUTPUTS_DIR = "/home/saptpurk/embeddings-noise-eliminators/outputs"
REPO_ROOT_OVERRIDE = "/home/saptpurk/embeddings-noise-eliminators"
HF_TOKEN_OVERRIDE = None     # set non-None only when running gated models locally


In [ ]:
# Apply papermill parameters to environment (no-op when env vars already set)
import os
os.environ.setdefault("DATASET", DATASET)
os.environ.setdefault("MODELS_TO_RUN", MODELS)
os.environ.setdefault("OUTPUTS_DIR", OUTPUTS_DIR)
os.environ.setdefault("REPO_ROOT", REPO_ROOT_OVERRIDE)
if HF_TOKEN_OVERRIDE:
    os.environ["HF_TOKEN"] = HF_TOKEN_OVERRIDE


In [1]:
import os, warnings
from pathlib import Path
import pandas as pd

ROOT = Path(os.environ.get('V4_WORK_DIR',
    '/home/saptpurk/embeddings-noise-eliminators_work'))
out_dir = ROOT / 'v4_exp16_label_noise'
out_dir.mkdir(parents=True, exist_ok=True)

import sys; sys.path.insert(0, str(Path('..').resolve()))
try:
    from common.label_noise import (build_high_confidence_subset,
                                      refit_probes_on_subset)
except Exception as e:
    warnings.warn(f'label_noise import failed: {e}')
    build_high_confidence_subset = None
    refit_probes_on_subset = None

if build_high_confidence_subset is None:
    pd.DataFrame({'status': ['module_absent']}).to_parquet(
        out_dir / 'exp16_manifest.parquet', index=False)
else:
    subset = build_high_confidence_subset(dataset='mimic')
    rows = refit_probes_on_subset(subset, exps=('exp01', 'exp02', 'exp06'))
    out = pd.DataFrame(rows)
    out.to_parquet(out_dir / 'exp16_label_noise.parquet', index=False)
    print(f'wrote {len(out)} rows -> {out_dir / "exp16_label_noise.parquet"}')


/tmp/ipykernel_867052/2763348684.py:15: UserWarning: label_noise import failed: No module named 'common.label_noise'
  warnings.warn(f'label_noise import failed: {e}')
